# Introduction to Statistical Learning
## Chapter 5 exercises: Resampling methods

In [3]:
# Usual imports
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

import statsmodels.api as sm
from ISLP import load_data
from ISLP.models import (ModelSpec as MS,
                         summarize,
                         poly)

In [21]:
# 
from sklearn.model_selection import train_test_split
from functools import partial
from sklearn.model_selection import \
     (cross_validate,
      KFold,
      ShuffleSplit)
from sklearn.base import clone

# We need the following wrapper function from ISLP
from ISLP.models import sklearn_sm

# 
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
import statsmodels.formula.api as smf 

#### Exercise 5

We can now estimate the error of the logistic regression we performed on the `Default` dataset from Chapter 4.

In [9]:
# Set random seed and load data
np.random.seed(3453)
Default = load_data("Default")
Default

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879
...,...,...,...,...
9995,No,No,711.555020,52992.378914
9996,No,No,757.962918,19660.721768
9997,No,No,845.411989,58636.156984
9998,No,No,1569.009053,36669.112365


In [13]:
# We first need to turn default into a boolean/int

Default['default_yes'] = (Default['default'] == 'Yes').astype(int)

In [16]:
# (a) Fit a logistic regression model - we use statsmodels
# Use statsmodels formula api instead of design matrix!

formula = 'default_yes ~ income + balance'
model = smf.logit(formula, data=Default)
results = model.fit()
results.summary()

Optimization terminated successfully.
         Current function value: 0.078948
         Iterations 10


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:            default_yes   No. Observations:                10000
Model:                          Logit   Df Residuals:                     9997
Method:                           MLE   Df Model:                            2
Date:                Wed, 30 Apr 2025   Pseudo R-squ.:                  0.4594
Time:                        17:59:48   Log-Likelihood:                -789.48
converged:                       True   LL-Null:                       -1460.3
Covariance Type:            nonrobust   LLR p-value:                4.541e-292
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -11.5405      0.435    -26.544      0.000     -12.393     -10.688
income      2.081e-05   4.99e-06      4.174      0.000     1.1e-05    3.06e-05
balance        0.0056      0.000     24.835      0.000       0.005       0.006
==============================================================================

Possibly complete quasi-separation: A fraction 0.14 of observations can be
perfectly predicted. This might indicate that there is complete
quasi-separation. In this case some parameters will not be identified.
"""

In [18]:
# Let's also do it with sklearn! (to check coeffs agree)
lr = LogisticRegression(C=1e6, tol=1e-6)
X = Default[['income', 'balance']]
y = Default['default_yes']
model = lr.fit(X, y)
model.coef_

array([[2.08089755e-05, 5.64710295e-03]])

In [28]:
# (b) Using validation set approach to estimate test error
# Do this in a function, so we can repeat it for part (c)!

def validation_approach(seed):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=seed)

    model = lr.fit(X_train, y_train)

    y_probability = model.predict(X_test)

    y_pred = (y_probability > 0.5)

    cm = confusion_matrix(y_test, y_pred)
    print(cm)

    error = 1 - accuracy_score(y_test, y_pred)
    print(f"Validation set error: {error:.3f}")
    

validation_approach(1232)

validation_approach(132)

validation_approach(123)

[[4814   28]
 [ 109   49]]
Validation set error: 0.027
[[4809   20]
 [ 118   53]]
Validation set error: 0.028
[[4825   20]
 [ 108   47]]
Validation set error: 0.026


The validation set errors show low variability across runs, indicating stable overall classification performance. However, other metrics—such as the false positive rate and true positive rate—exhibit more fluctuation, suggesting differences in how specific classes are being predicted.

In [35]:
# (d) now consider adding dummy variable for student

X = pd.get_dummies(Default[['income', 'balance', 'student']], drop_first=True)
y = Default['default_yes']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=3349)

model = lr.fit(X_train, y_train)

y_probability = model.predict(X_test)
y_pred = (y_probability > 0.5)

error = 1 - accuracy_score(y_test, y_pred)
print(f"Validation set error: {error:.3f}")

Validation set error: 0.027


The Validation set error stays fairly similar to the previous validation set errors, so including a dummy variable for student likely does not decrease the test error rate.

#### Exercise 6

In [39]:
# (a) We calculated these in Ex 5 to be 4.99e-06 and 0.
# (b) Wire bootstrap function

def boot_fn(default, idx):
    model = smf.logit(formula, data=default.iloc[idx]).fit(disp=False)
    return np.array([model.params[1],  model.params[2]])


def boot_SE(func,
            D,
            n=None,
            B=1000,
            seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0, 0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index,
                         n,
                         replace=True)
        value = func(D, idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

coeff_SE = boot_SE(boot_fn, Default, B=100, seed=99)
coeff_SE


C:\Users\gavin\AppData\Local\Temp\ipykernel_27764\3754852712.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return np.array([model.params[1],  model.params[2]])
C:\Users\gavin\AppData\Local\Temp\ipykernel_27764\3754852712.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return np.array([model.params[1],  model.params[2]])
C:\Users\gavin\AppData\Local\Temp\ipykernel_27764\3754852712.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc

array([4.65358407e-06, 2.31561951e-04])